# BigQuery ML: Health Professional Shortage Area analytics (Kaggle run)

How well do context features (HHS region, state, rural status, designation type, population type,
metropolitan indicator) predict a US Health Professional Shortage Area's priority score, **without**
the inputs HRSA uses to compute that score?

This notebook runs the same feature SQL as `queries/01_features.sql` with `google.cloud.bigquery`, then fits the
same two model types as the BigQuery ML scripts (linear and logistic regression, scikit-learn here) and compares
them with mean and majority baselines on the same hash-based 80/20 split.

Kaggle settings: **Internet on** (the SQL is read from the GitHub repo). If BigQuery access fails, attach a Google
Cloud project under *Add-ons > Google Cloud Services*.

In [ ]:
from __future__ import annotations
import json, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, mean_absolute_error, mean_squared_error,
                             r2_score, roc_auc_score, roc_curve)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

REPO_RAW = "https://raw.githubusercontent.com/MHKhan50/enterprise-bigquery-ml-healthcare/main/queries/"
TABLES = ["hpsa_primary_care", "hpsa_dental_health", "hpsa_mental_health"]
FEATURES = ["hhs_region", "state", "rural_status", "designation_type", "population_type", "metro_indicator"]
SOURCE_COLUMNS = ["HPSA_Name", "HPSA_Score", "HPSA_Withdrawn_Date", "Status_Description",
                  "Primary_HHS_Region_Name", "Common_State_Abbr", "Rural_Status_Desc", "Type_Desc",
                  "HPSA_Pop_Type_Desc", "Metropolitan_Indicator_Desc"]
SEED = 42

In [ ]:
MARKER = "-- SELECT_STARTS_HERE"

def feature_select(sql_text: str, table: str) -> str:
    """Return the SELECT part of 01_features.sql, pointed at one HPSA table."""
    if MARKER not in sql_text:
        raise ValueError("marker line not found in 01_features.sql")
    body = sql_text.split(MARKER, 1)[1].strip().rstrip(";")
    return body.replace("sdoh_hrsa_shortage_areas.hpsa_primary_care", "sdoh_hrsa_shortage_areas." + table)


def _bootstrap(test_idx_n: int, stat, n_boot: int = 1000, seed: int = SEED):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, test_idx_n, test_idx_n)
        try:
            vals.append(stat(idx))
        except ValueError:  # one class only in a resample
            continue
    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]


def run_models(df: pd.DataFrame) -> dict:
    """Fit linear (hpsa_score) and logistic (high_need) models on split=train, score on split=test."""
    df = df.copy()
    for c in FEATURES:
        df[c] = df[c].fillna("missing").astype(str)
    tr, te = df[df.split == "train"], df[df.split == "test"]
    enc = lambda: ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES)])

    lin = make_pipeline(enc(), LinearRegression()).fit(tr[FEATURES], tr.hpsa_score)
    y, p = te.hpsa_score.to_numpy(float), lin.predict(te[FEATURES])
    base = np.full_like(y, tr.hpsa_score.mean())
    n = len(y)

    log = make_pipeline(enc(), LogisticRegression(max_iter=2000)).fit(tr[FEATURES], tr.high_need)
    yc = te.high_need.to_numpy(int)
    prob = log.predict_proba(te[FEATURES])[:, 1]
    pred = (prob >= 0.5).astype(int)
    majority = int(tr.high_need.mean() >= 0.5)

    return {
        "areas_train": int(len(tr)), "areas_test": int(n),
        "high_need_share_test": float(yc.mean()),
        "linear": {
            "mae": float(mean_absolute_error(y, p)),
            "rmse": float(np.sqrt(mean_squared_error(y, p))),
            "r2": float(r2_score(y, p)),
            "mae_gain_vs_mean_ci95": _bootstrap(n, lambda i: np.abs(y[i] - base[i]).mean() - np.abs(y[i] - p[i]).mean()),
        },
        "mean_baseline": {"mae": float(mean_absolute_error(y, base)),
                          "rmse": float(np.sqrt(mean_squared_error(y, base)))},
        "logistic": {
            "accuracy": float(accuracy_score(yc, pred)),
            "f1": float(f1_score(yc, pred, zero_division=0)),
            "roc_auc": float(roc_auc_score(yc, prob)),
            "roc_auc_ci95": _bootstrap(n, lambda i: roc_auc_score(yc[i], prob[i])),
        },
        "majority_baseline": {"accuracy": float((yc == majority).mean()), "roc_auc": 0.5},
        "_roc": [a.tolist() for a in roc_curve(yc, prob)[:2]],
    }

In [ ]:
from google.cloud import bigquery
client = bigquery.Client()

def fetch(name: str) -> str:
    return urllib.request.urlopen(REPO_RAW + name, timeout=30).read().decode("utf-8")

# Step 0: check the real columns before using them
schema = client.query(fetch("00_check_columns.sql")).to_dataframe()
for t in TABLES:
    have = set(schema.loc[schema.table_name == t, "column_name"])
    missing = [c for c in SOURCE_COLUMNS if c not in have]
    print(t, "columns:", len(have), "| missing:", missing or "none")
    if missing:
        print("  real columns:", sorted(have))
assert all(set(SOURCE_COLUMNS) <= set(schema.loc[schema.table_name == t, "column_name"]) for t in TABLES), \
    "A needed column is missing: send this output so the SQL can be fixed."

In [ ]:
features_sql = fetch("01_features.sql")
results, frames = {}, {}
for t in TABLES:
    df = client.query(feature_select(features_sql, t)).to_dataframe()
    frames[t] = df
    print(t, "areas:", len(df), "| status filter kept rows:", len(df) > 0)
    results[t] = run_models(df)
    r = results[t]
    print(f"  linear  MAE {r['linear']['mae']:.2f} vs mean {r['mean_baseline']['mae']:.2f}"
          f"  (gain 95% CI {r['linear']['mae_gain_vs_mean_ci95']})  R2 {r['linear']['r2']:.3f}")
    print(f"  logistic AUC {r['logistic']['roc_auc']:.3f} (95% CI {r['logistic']['roc_auc_ci95']})"
          f"  acc {r['logistic']['accuracy']:.3f} vs majority {r['majority_baseline']['accuracy']:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(TABLES))
ax[0].bar(x - 0.2, [results[t]["mean_baseline"]["mae"] for t in TABLES], 0.4, label="mean baseline", color="#9aa5b1")
ax[0].bar(x + 0.2, [results[t]["linear"]["mae"] for t in TABLES], 0.4, label="linear regression", color="#1f6feb")
ax[0].set_xticks(x, [t.replace("hpsa_", "") for t in TABLES]); ax[0].set_ylabel("test MAE (score points)")
ax[0].set_title("HPSA score error, lower is better"); ax[0].legend()
for t in TABLES:
    fpr, tpr = results[t]["_roc"]
    ax[1].plot(fpr, tpr, label=f"{t.replace('hpsa_', '')} AUC {results[t]['logistic']['roc_auc']:.2f}")
ax[1].plot([0, 1], [0, 1], "--", color="#9aa5b1", label="chance")
ax[1].set_xlabel("false positive rate"); ax[1].set_ylabel("true positive rate")
ax[1].set_title("High-need areas (top quarter of score)"); ax[1].legend()
plt.tight_layout(); plt.savefig("hpsa_results.png", dpi=150); plt.show()

clean = {t: {k: v for k, v in r.items() if k != "_roc"} for t, r in results.items()}
json.dump(clean, open("results.json", "w"), indent=2)
print(json.dumps(clean, indent=2))